# PID Controllers (PID_1 and PID_2)

**Transfer function (PID_1, crossed):**
G_c(s) = [α₁(s+1)² − α₂·β·ρ] / [(s+1)³ − β·k·(s+1)]
       = [α₁s² + 2α₁s + (α₁−α₂·β·ρ)] / [(s+1)·((s+1)²−β·k)]

**Key gains:**
- K_D_eff = α₁          (derivative-like coefficient)
- K_I_eff = α₁ − α₂·β·ρ (integral-like coefficient, proportional numerator at DC)

**Steady-state error:**
e_ss = r / (1 + θ₁θ₂·(α₁−α₂·β·ρ)/(1−β·k))

**Zero SS error requires β·k = 1** (not automatic — depends on parameter choice).
At default β=0.3, k=0.3: β·k = 0.09, so there IS steady-state error.

**PID_1 (crossed):** derivative reduces K_I_eff → tighter damping.
**PID_2 (uncrossed):** derivative boosts K_I_eff → faster but less stable.

**Stability boundary (5th-order characteristic polynomial + integrator):** α₁·θ₁·θ₂ < **6**.

## Setup

In [27]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
import numpy as np
import sympy as sp
import warnings
warnings.filterwarnings('ignore')
from bokeh.io import output_notebook, show

import biomolecular_controllers as bc
from biomolecular_controllers.gain import pid_stability_boundary

print('✓ Imports successful!')
from biomolecular_controllers.figure_saver import save_fig
from biomolecular_controllers.figure_gifs import save_gif


output_notebook()



✓ Imports successful!


Loading BokehJS ...

In [29]:
output_notebook()

runner  = bc.SimulationRunner()
calc    = bc.MetricsCalculator()
stab    = bc.StabilityAnalyzer()
plotter = bc.VisualizationPipeline()
swiffer = bc.SensitivityAnalyzer()


print('✓ Tools initialized!')

Loading BokehJS ...

✓ Tools initialized!


## Parameter Sweep

In [30]:
# Set fixed params such that contribution to any gain(s) not being investigated is ~0.1 or less
# Effective Gain K_p = 2*alpha1
# Effective K_d = alpha1 
# Effective K_i = alpha1 - alpha2*beta*rho  (when beta*k=1)

alpha1, alpha2, beta, k, rho, theta1, theta2 = sp.symbols(
    'alpha1 alpha2 beta k rho theta1 theta2', positive=True
)

q  = beta * k
K1 = alpha1 * theta1 * theta2
K2 = alpha2 * beta * rho * theta1 * theta2

# Routh-Hurwitz conditions, each must be > 0
rh_conditions = {
    'P_LPF': 8 - K1,              # upper bound, consistent across all controllers
    'PI':    1 - q + K1 - K2      # q=1 case, lower bound on K1
}

base_params = {
    theta1: 1.0,   # theta1*theta2 = 1 so gains read directly
    theta2: 1.0,
    beta:   1.0,   # beta=1 decouples beta*k=k and beta*alpha2=alpha2
    k:      0.999, # beta*k=1 nearly exactly, integral action active
    alpha1: 1.0,   # fix so KP,KD baseline known while rho varies KI
    alpha2: 1.0,   # initialization, overridden when sweeping
    rho:    1.0,   # scales integral gain alpha2*beta*rho
}

# example with simulation parameters
# sweep alpha1, fix everything else
lower, upper = swiffer.get_feasible_range(rh_conditions, alpha1, base_params)

inside, out_low, out_high, combined = swiffer.build_sweep(lower, upper)
print(f"Feasible range for alpha2 with small excursion above/below stable regime: ({lower:.4f}, {upper:.4f})")
print(f"Inside:  {np.round(inside, 3)}")
print(f"Below:   {np.round(out_low, 3)}")
print(f"Above:   {np.round(out_high, 3)}")

  P_LPF: upper bound at 8.0000
  PI: lower bound at 0.9990
Feasible range for alpha2 with small excursion above/below stable regime: (0.9990, 8.0000)
Inside:  [1.349 2.249 3.149 4.049 4.95  5.85  6.75  7.65 ]
Below:   [-0.051]
Above:   [9.05]


## Constrained Sweeps

In [31]:
rho = sp.Symbol('rho', positive=True)

KP_expr_PID  = 2 * alpha1 * theta1 * theta2
KD_expr_PID  = alpha1 * theta1 * theta2
KI_expr_PID  = (alpha1 - alpha2*beta*rho) * theta1 * theta2
INT_expr_PID = beta * k   # =1 for integral action

fixed_PID = {theta1: 1.0, theta2: 1.0, alpha1: 2.0}

# note KP and KD are both tied to alpha1 so you only have
# two free constraints: KI and beta*k=1
# meaning you can only solve for two dependent params
# natural choice is to sweep alpha2, solve for beta and k (or rho)

sweep_alpha2_PID = swiffer.make_constrained_sweep(
    free_param_vals  = np.linspace(0.1, 1.5, 8),
    free_param_sym   = alpha2,
    gain_constraints = {
        'integral_action': (INT_expr_PID, 1.0),  # beta*k=1
        'integral_gain':   (KI_expr_PID,  1.0),  # KI fixed
    },
    all_params_sym   = [alpha2, beta, k, rho],
    fixed_params     = {**fixed_PID, rho: 1.0}   # fix rho, solve beta and k
)

Symbolic solutions for dependent params:
  beta = α₁⋅θ₁⋅θ₂ - 1.0
──────────────
  α₂⋅ρ⋅θ₁⋅θ₂  
  k =   α₂⋅ρ⋅θ₁⋅θ₂  
──────────────
α₁⋅θ₁⋅θ₂ - 1.0


## Part 1: Sweep α₁ (Vary Gain) — PID_1

Hold β (derivative) and k (integral) fixed. Sweep α₁ to vary closed-loop gain G = α₁·θ₁·θ₂.
Stability requires G < 6.  Expect: zero SS error, overshoot decreasing as G decreases.

In [32]:
# Sweep alpha_1 
alpha_vals = inside

# Step input
ref_low  = bc.DEFAULT_PARAMS['PID_1']['ref']
ref_high = 50.0
t_step   = 150.0
t_span   = (0, 600)

# Effective integral-like gain numerator: K_I_eff_num = (α₁ − α₂·β·ρ)·θ₁θ₂
print(f'Sweeping α₁: {alpha_vals.min():.2f} → {alpha_vals.max():.2f}')
print(f'Step input: {bc.DEFAULT_PARAMS["PID_1"]["ref"]} → 50.0 at t=150.0')

Sweeping α₁: 1.35 → 7.65
Step input: 10.0 → 50.0 at t=150.0


In [33]:
trajectories_dict   = {}
gains_list          = []
ss_errors_list      = []
ss_stds_list        = []
settling_times_list = []
overshoots_list     = []
rise_times_list     = []

params = {
    'theta_1': 1.0,
    'theta_2': 1.0,
    'beta':    1.0,
    'k':       1.0 + 1e-2,
    'alpha_2': 0.1563,
    'alpha_1': 1.0,
    'rho': 1.0,
    
}
print('Running PID_1 simulations...')

for alpha in alpha_vals:
    current_params = {**params, 'alpha_1': alpha}
    perturbations = [
        {'time': t_step, 'type': 'parameter', 'param': 'ref', 'value': ref_high}
    ]
    try:
        result = runner.run_with_perturbations(
            model='PID_1', t_span=t_span, points=1200,
            perturbations=perturbations, params=current_params
        )
        time = np.asarray(result['time'],        dtype=float)
        y    = np.asarray(result['states']['y'], dtype=float)
        ref  = np.where(time < t_step, ref_low, ref_high)

        # Effective integral gain: K_I_eff = (α₁−α₂·β·ρ)·θ₁θ₂ / (β·k−1)
        K_P_eff_num = bc.gain.gain_pid1(current_params)

        # Use actual SS value for rise_time (SS error may be non-zero when β·k ≠ 1)
        y_ss = float(np.mean(y[-20:]))

        os_r = calc.overshoot(time, y, ref=ref_high, pert_start=t_step, pert_end=t_span[1])
        st_r = calc.settling_time(time, y, ref=ref_high, pert_start=t_step, pert_end=t_span[1])
        rt_r = calc.rise_time(time, y, ref=y_ss, pert_start=t_step, pert_end=t_span[1])
        ss_r = calc.steady_state(time,y, pert_start=t_step, pert_end=t_span[1])

        trajectories_dict[alpha] = {'time': time, 'y': y, 'ref': ref}
        gains_list.append(K_P_eff_num)
        ss_errors_list.append(abs(ss_r['ss_value'] - ref_high))
        ss_stds_list.append(ss_r['ss_std'])
        settling_times_list.append(st_r['settling_time'] if st_r['settled'] else np.nan)
        overshoots_list.append(os_r['magnitude'])
        rise_times_list.append(rt_r['rise_time'] if rt_r['valid'] else np.nan)
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f'  α₁={alpha:.2f}: failed ({str(e)[:50]})')
        gains_list.append(bc.gain.gain_pid1(params))
        ss_errors_list.append(np.nan)
        ss_stds_list.append(np.nan)
        settling_times_list.append(np.nan)
        overshoots_list.append(np.nan)
        rise_times_list.append(np.nan)

gains          = np.array(gains_list)
ss_errors      = np.array(ss_errors_list)
ss_stds        = np.array(ss_stds_list)
settling_times = np.array(settling_times_list)
overshoots     = np.array(overshoots_list)
rise_times     = np.array(rise_times_list)

print(f'✓ Completed {np.sum(~np.isnan(ss_errors))}/{len(alpha_vals)} simulations')
print(f'SS errors: {ss_errors}')

Running PID_1 simulations...
✓ Completed 8/8 simulations
SS errors: [ 0.42273089  0.24004073  0.16760572  0.12875093  0.1044265   0.09222626
 21.37091426 24.81312584]


## Plot 1: Steady-State Error vs K_I_eff_num

**e_ss = r / (1 + θ₁θ₂·(α₁−α₂·β·ρ)/(1−β·k))**

With β=0.3, k=0.3: β·k=0.09, so SS error is non-zero.
x-axis: K_I_eff_num = (α₁−α₂·β·ρ)·θ₁θ₂  (the numerator effective gain).
Zero SS error only when β·k = 1.

In [34]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=np.array(alpha_vals),
    trajectories=trajectories_dict,
    metric_name='Steady State Error',
    gain_vals=gains,
    metric_vals=ss_errors,
    title='PID_1: Steady-State Error vs K_I_eff_num  [e_ss = r/(1+K_I_eff_num/(1−β·k))]',
    x_label='K_I_eff_num = (α₁−α₂·β·ρ)·θ₁θ₂',
    x_scale='linear',
    y_scale='linear',
    return_sources=True,
)

show(layout)
#save_fig(layout, 'PID_1', 'steady_state_error', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PID_1', 'steady_state_error')


[0, 1, 2, 3, 4, 5, 6, 7]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pid_1_steady_state_error.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pid_1_steady_state_error.gif')

## Plot 2: Settling Time vs Gain

In [35]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=np.array(alpha_vals),
    trajectories=trajectories_dict,
    metric_name='Settling Time',
    gain_vals=gains,
    metric_vals=settling_times,
    title='PID_1: Settling Time vs K_I_eff_num',
    x_label='K_I_eff_num = (α₁−α₂·β·ρ)·θ₁θ₂',
    x_scale='linear',
    y_scale='linear',
    return_sources=True,
)

show(layout)
#save_fig(layout, 'PID_1', 'settling_time', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PID_1', 'settling_time')


[0, 1, 2, 3, 4, 5, 6, 7]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pid_1_settling_time.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pid_1_settling_time.gif')

## Plot 3: Overshoot vs Gain

In [36]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=np.array(alpha_vals),
    trajectories=trajectories_dict,
    metric_name='Overshoot',
    gain_vals=gains,
    metric_vals=overshoots,
    title='PID_1: Overshoot vs K_I_eff_num',
    x_label='K_I_eff_num = (α₁−α₂·β·ρ)·θ₁θ₂',
    x_scale='linear',
    y_scale='linear',
    return_sources=True,
)

show(layout)
#save_fig(layout, 'PID_1', 'overshoot', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PID_1', 'overshoot')


[0, 1, 2, 3, 4, 5, 6, 7]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pid_1_overshoot.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pid_1_overshoot.gif')

## Plot 4: Rise Time vs Gain

In [37]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=np.array(alpha_vals),
    trajectories=trajectories_dict,
    metric_name='Rise Time',
    gain_vals=gains,
    metric_vals=rise_times,
    title='PID_1: Rise Time vs K_I_eff_num  (measured to actual SS value)',
    x_label='K_I_eff_num = (α₁−α₂·β·ρ)·θ₁θ₂',
    x_scale='linear',
    y_scale='linear',
    return_sources=True,
)

show(layout)
#save_fig(layout, 'PID_1', 'rise_time', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PID_1', 'rise_time')


[0, 1, 2, 3, 4, 5, 6, 7]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pid_1_rise_time.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pid_1_rise_time.gif')

## Part 2: Stability Diagrams (α₁ vs θ₁)

Boundary: θ₁ = 6 / (α₁·θ₂) — same formula and G_crit as PI.
The derivative (β) does not change this first-order boundary, but numerically it broadens stability margins.

In [38]:
# PID_1 Parameter stability diagram
alpha_range = np.linspace(0.1, 10, 400)
theta_range = np.linspace(0.1, 10, 400)
theta_2_fixed = 1.0

boundary_fn = bc.gain.pid_stability_boundary(theta_2=theta_2_fixed)

p = plotter.plot_stability_diagram(
    x_vals=alpha_range,
    y_vals=theta_range,
    stable_condition=lambda A, T: (A * T * theta_2_fixed < 6),
    boundary_fns=[
        {
            'fn':    boundary_fn,
            'label': 'PID_1 boundary  (G=6)',
            'color': 'black',
            'dash':  'dashed',
            'line_width': 2.0,
        },
    ],
    x_name='α₁',
    y_name='θ₁',
    title=f'PID_1 Parameter Stability Diagram  (θ₂={theta_2_fixed})',
)
p.scatter([-999], [-999], marker="square", size=14,
          fill_color="#8ECF8E", fill_alpha=0.65,
          line_color=None, legend_label="Stable")
p.scatter([-999], [-999], marker="square", size=14,
          fill_color="#C8C8C8", fill_alpha=0.65,
          line_color=None, legend_label="Unstable")

save_fig(p, 'PID_1', 'stability_diagram', fmt='png', show=True)

  Saved PNG  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pid_1_stability_diagram.png


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pid_1_stability_diagram.png')

## Part 3: Root Locus (Dominant Eigenvalue)

Compare PID_1 vs PID_2 as α₁ is varied.  The derivative action shifts the dominant pole trajectory compared to pure PI.

In [39]:
root_alpha_vals = np.linspace(0.1, 5.5, 12)

eigenvalue_paths = {}
for model_key in ['PID_1', 'PID_2']:
    reals, imags, gs = [], [], []
    for val in root_alpha_vals:
        p_params = {
            'alpha_1': val,
            'alpha_2': 2.0,
            'theta_1': 1.0,
            'theta_2': 1.0,
            'beta':    1.0,
            'k':       1.001,
            'rho':     1.0,
        }
        try:
            res  = stab.analyze_stability(model=model_key, params=p_params, steady_state_time=100.0)
            eigs = res.eigenvalues
            dom  = eigs[np.argmax(np.real(eigs))]
            reals.append(float(np.real(dom)))
            imags.append(float(np.imag(dom)))
            gs.append(res.gain)
        except Exception:
            reals.append(np.nan); imags.append(np.nan); gs.append(np.nan)
    eigenvalue_paths[model_key] = {
        'real': np.array(reals),
        'imag': np.array(imags),
        'gain': np.array(gs),
    }

fig = plotter.plot_root_locus(
    eigenvalue_paths,
    title='PID Controllers Root Locus: Dominant Eigenvalue as α₁ varies'
)
show(fig)
save_fig(fig, 'PID_controllers', 'root_locus', fmt='png', show=True)

  Saved PNG  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pid_controllers_root_locus.png


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pid_controllers_root_locus.png')